In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [2]:
df = pd.read_csv("../data/processed/telco_customer_churn_cleaned.csv")

print("Shape:", df.shape)
df.head()

Shape: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [3]:
df = df.drop(columns=["customerID"])

print("Shape after removing customerID:", df.shape)

Shape after removing customerID: (7043, 20)


In [4]:
df["Churn"] = df["Churn"].map({
    "No": 0,
    "Yes": 1
})

df["Churn"].value_counts()

Churn
0    5174
1    1869
Name: count, dtype: int64

In [5]:
X = df.drop(columns=["Churn"])
y = df["Churn"]

print("Features shape:", X.shape)
print("Target shape:", y.shape)

Features shape: (7043, 19)
Target shape: (7043,)


In [6]:
categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()
numerical_cols = X.select_dtypes(exclude=["object"]).columns.tolist()

print("Categorical features:")
print(categorical_cols)

print("\nNumerical features:")
print(numerical_cols)

Categorical features:
['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']

Numerical features:
['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']


In [7]:
X_encoded = pd.get_dummies(
    X,
    columns=categorical_cols,
    drop_first=True,
    dtype=int
)

print("Encoded feature shape:", X_encoded.shape)

Encoded feature shape: (7043, 30)


In [8]:
X_encoded.head()

,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,gender_Male,Partner_Yes,Dependents_Yes,PhoneService_Yes,MultipleLines_No phone service,MultipleLines_Yes,...,StreamingTV_No internet service,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaperlessBilling_Yes,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,0,1,29.85,29.85,0,1,0,0,1,0,...,0,0,0,0,0,0,1,0,1,0
1,0,34,56.95,1889.50,1,0,0,1,0,0,...,0,0,0,0,1,0,0,0,0,1
2,0,2,53.85,108.15,1,0,0,1,0,0,...,0,0,0,0,0,0,1,0,0,1
3,0,45,42.30,1840.75,1,0,0,0,1,0,...,0,0,0,0,1,0,0,0,0,0
4,0,2,70.70,151.65,0,0,0,1,0,0,...,0,0,0,0,0,0,1,0,1,0


In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts())

print("\nTesting target distribution:")
print(y_test.value_counts())

Training features: (5634, 30)
Testing features: (1409, 30)

Training target distribution:
Churn
0    4139
1    1495
Name: count, dtype: int64

Testing target distribution:
Churn
0    1035
1     374
Name: count, dtype: int64


In [10]:
numeric_features = [
    "SeniorCitizen",
    "tenure",
    "MonthlyCharges",
    "TotalCharges"
]

In [11]:
numeric_indices = [
    X_train.columns.get_loc(col)
    for col in numeric_features
]

numeric_indices

[0, 1, 2, 3]

In [12]:
scaler = StandardScaler()

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[numeric_features] = scaler.fit_transform(
    X_train[numeric_features]
)

X_test_scaled[numeric_features] = scaler.transform(
    X_test[numeric_features]
)

In [13]:
print("Training numerical feature means:")
print(X_train_scaled[numeric_features].mean())

print("\nTraining numerical feature standard deviations:")
print(X_train_scaled[numeric_features].std())

Training numerical feature means:
SeniorCitizen     7.440898e-17
tenure           -1.008935e-17
MonthlyCharges   -2.402527e-16
TotalCharges      2.522338e-17
dtype: float64

Training numerical feature standard deviations:
SeniorCitizen     1.000089
tenure            1.000089
MonthlyCharges    1.000089
TotalCharges      1.000089
dtype: float64


In [14]:
print("Unscaled training data:", X_train.shape)
print("Unscaled testing data:", X_test.shape)

print("Scaled training data:", X_train_scaled.shape)
print("Scaled testing data:", X_test_scaled.shape)

print("\nMissing values:")
print("X_train:", X_train.isnull().sum().sum())
print("X_test:", X_test.isnull().sum().sum())
print("X_train_scaled:", X_train_scaled.isnull().sum().sum())
print("X_test_scaled:", X_test_scaled.isnull().sum().sum())

Unscaled training data: (5634, 30)
Unscaled testing data: (1409, 30)
Scaled training data: (5634, 30)
Scaled testing data: (1409, 30)

Missing values:
X_train: 0
X_test: 0
X_train_scaled: 0
X_test_scaled: 0


In [15]:
X_train.to_csv(
    "../data/processed/X_train.csv",
    index=False
)

X_test.to_csv(
    "../data/processed/X_test.csv",
    index=False
)

X_train_scaled.to_csv(
    "../data/processed/X_train_scaled.csv",
    index=False
)

X_test_scaled.to_csv(
    "../data/processed/X_test_scaled.csv",
    index=False
)

y_train.to_csv(
    "../data/processed/y_train.csv",
    index=False
)

y_test.to_csv(
    "../data/processed/y_test.csv",
    index=False
)

print("Feature-engineered datasets saved successfully.")

Feature-engineered datasets saved successfully.


In [16]:
print("Feature Engineering Complete")
print("-" * 40)
print(f"Original cleaned rows : {len(df)}")
print(f"Training rows         : {len(X_train)}")
print(f"Testing rows          : {len(X_test)}")
print(f"Final feature count   : {X_train.shape[1]}")
print(f"Categorical features  : {len(categorical_cols)}")
print(f"Numerical features    : {len(numerical_cols)}")

Feature Engineering Complete
----------------------------------------
Original cleaned rows : 7043
Training rows         : 5634
Testing rows          : 1409
Final feature count   : 30
Categorical features  : 15
Numerical features    : 4
